In [1]:
import copy
import math

X = "X"
O = "O"
EMPTY = None

# --- CẤU HÌNH TRÒ CHƠI ---
N = 5             # Kích thước bàn cờ N x N
WIN_COUNT = 4     # Số quân liên tiếp để thắng (ví dụ: 5x5 cần 4, 10x10 cần 5)
MAX_DEPTH = 3     # Độ sâu tối đa AI tìm kiếm (Tăng lên sẽ thông minh hơn nhưng chậm hơn)

def initial_state(n):
    return [[EMPTY for _ in range(n)] for _ in range(n)]

def player(turn_count, user_char):
    ai_char = O if user_char == X else X
    return user_char if turn_count % 2 == 0 else ai_char

def actions(board):
    return {(i, j) for i in range(N) for j in range(N) if board[i][j] == EMPTY}

def result(board, action, p_char):
    i, j = action
    new_board = [row[:] for row in board] # copy nhanh hơn deepcopy cho mảng 2D
    new_board[i][j] = p_char
    return new_board

def check_winner(board):
    """Kiểm tra người thắng cho ma trận N x N"""
    for r in range(N):
        for c in range(N):
            if board[r][c] is None: continue
            p = board[r][c]
            # Ngang, Dọc, Chéo xuôi, Chéo ngược
            directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
            for dr, dc in directions:
                count = 0
                for i in range(WIN_COUNT):
                    nr, nc = r + dr * i, c + dc * i
                    if 0 <= nr < N and 0 <= nc < N and board[nr][nc] == p:
                        count += 1
                    else:
                        break
                if count == WIN_COUNT: return p
    return None

def terminal(board):
    return check_winner(board) is not None or all(all(cell is not None for cell in row) for row in board)

# --- HÀM ĐÁNH GIÁ (HEURISTIC) ---
def evaluate(board, user_char):
    """
    Hàm đánh giá khi chưa tới đích.
    Tính điểm dựa trên các chuỗi quân cờ tiềm năng.
    """
    win = check_winner(board)
    ai_char = O if user_char == X else X
    if win == ai_char: return 10000
    if win == user_char: return -10000

    # Ở đây có thể thêm logic đếm các chuỗi 2, 3 quân để AI chơi giỏi hơn
    return 0

# --- THUẬT TOÁN ALPHA-BETA ---
def alpha_beta_search(board, user_char):
    if terminal(board): return None
    ai_char = O if user_char == X else X

    alpha = -math.inf
    beta = math.inf
    best_move = None
    best_val = -math.inf

    # Ưu tiên các ô gần trung tâm để tìm kiếm hiệu quả hơn
    possible_moves = sorted(list(actions(board)), key=lambda x: abs(x[0]-N/2) + abs(x[1]-N/2))

    for action in possible_moves:
        val = min_value(result(board, action, ai_char), user_char, alpha, beta, 1)
        if val > best_val:
            best_val = val
            best_move = action
        alpha = max(alpha, best_val)
    return best_move

def max_value(state, user_char, alpha, beta, depth):
    if terminal(state) or depth >= MAX_DEPTH:
        return evaluate(state, user_char)
    v = -math.inf
    ai_char = O if user_char == X else X
    for action in actions(state):
        v = max(v, min_value(result(state, action, ai_char), user_char, alpha, beta, depth + 1))
        alpha = max(alpha, v)
        if beta <= alpha: break
    return v

def min_value(state, user_char, alpha, beta, depth):
    if terminal(state) or depth >= MAX_DEPTH:
        return evaluate(state, user_char)
    v = math.inf
    for action in actions(state):
        v = min(v, max_value(result(state, action, user_char), user_char, alpha, beta, depth + 1))
        beta = min(beta, v)
        if beta <= alpha: break
    return v

# --- GIAO DIỆN ---
def print_board(board):
    n = len(board)

    # In hàng chỉ số cột (0, 1, 2, 3...)
    # Dùng f-string với độ rộng 4 để căn giữa các số
    print("\n     " + "   ".join(f"{j}".center(1) for j in range(n)))

    # In đường kẻ ngang đầu tiên
    print("    " + "+---" * n + "+")

    for i, row in enumerate(board):
        # In chỉ số hàng và bắt đầu dòng nội dung
        row_str = f" {i:2} |" # i:2 giúp số hàng 0-9 và 10+ luôn thẳng hàng
        for cell in row:
            char = cell if cell else "."
            row_str += f" {char} |"
        print(row_str)

        # In đường kẻ ngang phân cách các hàng
        print("    " + "+---" * n + "+")

if __name__ == "__main__":
    N = int(input("Nhập kích thước ma trận N: "))
    WIN_COUNT = int(input(f"Nhập số quân liên tiếp để thắng (gợi ý: {min(N, 5)}): "))
    MAX_DEPTH = 3 if N <= 5 else 2 # Bàn cờ càng lớn độ sâu phải càng thấp để tránh treo máy

    board = initial_state(N)
    user_p = input("Chọn X hoặc O: ").upper()
    turn = 0

    while not terminal(board):
        print_board(board)
        curr_p = player(turn, user_p)

        if curr_p == user_p:
            print(f"\nLượt của bạn ({user_p})")
            try:
                r, c = map(int, input("Nhập hàng và cột (ví dụ: 1 2): ").split())
                if board[r][c] is not None: raise ValueError
                board[r][c] = user_p
            except:
                print("Ô không hợp lệ!")
                continue
        else:
            print("AI đang suy nghĩ...")
            move = alpha_beta_search(board, user_p)
            if move: board[move[0]][move[1]] = curr_p
        turn += 1

    print_board(board)
    res = check_winner(board)
    print(f"KẾT THÚC: {res} THẮNG!" if res else "KẾT THÚC: HÒA!")

Nhập kích thước ma trận N: 5
Nhập số quân liên tiếp để thắng (gợi ý: 5): 3
Chọn X hoặc O: x

     0   1   2   3   4
    +---+---+---+---+---+
  0 | . | . | . | . | . |
    +---+---+---+---+---+
  1 | . | . | . | . | . |
    +---+---+---+---+---+
  2 | . | . | . | . | . |
    +---+---+---+---+---+
  3 | . | . | . | . | . |
    +---+---+---+---+---+
  4 | . | . | . | . | . |
    +---+---+---+---+---+

Lượt của bạn (X)
Nhập hàng và cột (ví dụ: 1 2): 1
Ô không hợp lệ!

     0   1   2   3   4
    +---+---+---+---+---+
  0 | . | . | . | . | . |
    +---+---+---+---+---+
  1 | . | . | . | . | . |
    +---+---+---+---+---+
  2 | . | . | . | . | . |
    +---+---+---+---+---+
  3 | . | . | . | . | . |
    +---+---+---+---+---+
  4 | . | . | . | . | . |
    +---+---+---+---+---+

Lượt của bạn (X)
Nhập hàng và cột (ví dụ: 1 2): 1
Ô không hợp lệ!

     0   1   2   3   4
    +---+---+---+---+---+
  0 | . | . | . | . | . |
    +---+---+---+---+---+
  1 | . | . | . | . | . |
    +---+---+---+---+---+
